# Phase 3 - Prophet Demand Forecasting

Notebook này hoàn thiện Phase 3 với Prophet: từ tiền xử lý dữ liệu, huấn luyện, đánh giá, tuning đến xuất artifact cho bàn giao.

Mục tiêu:
- Dự báo doanh thu theo tháng cho 3 category lớn nhất
- Đánh giá bằng backtesting và MAPE / MAE / RMSE
- Tuning một bộ tham số Prophet cơ bản
- Lưu forecast và metrics ra file để dùng cho Phase 4

## 1. Thiết lập môi trường và import thư viện

Cài đặt và import `pandas`, `numpy`, `matplotlib`, `prophet`, `scikit-learn`; cấu hình seed và warning để chạy ổn định trong VS Code notebook.

In [ ]:
fact_transactions = pd.read_parquet(DATA_DIR / "fact_transactions.parquet")
dim_products = pd.read_parquet(DATA_DIR / "dim_products.parquet")

fact_transactions = fact_transactions.copy()
fact_transactions["date_key"] = fact_transactions["date_key"].astype(str)
fact_transactions["date"] = pd.to_datetime(fact_transactions["date_key"], format="%Y%m%d", errors="coerce")
fact_transactions = fact_transactions.dropna(subset=["date"])
fact_transactions = fact_transactions.sort_values(["date", "product_id"]) 
fact_transactions = fact_transactions.drop_duplicates(subset=["date", "product_id", "sale_price"], keep="last")

monthly_sales = (
    fact_transactions.assign(month=lambda df: df["date"].dt.to_period("M").dt.to_timestamp())
    .merge(dim_products[["product_id", "category"]], on="product_id", how="left")
    .groupby(["month", "category"], as_index=False)["sale_price"]
    .sum()
    .rename(columns={"month": "ds", "sale_price": "y"})
)

print(f"Monthly series rows: {len(monthly_sales):,}")
print(monthly_sales.head())

## 3. Khám phá dữ liệu nhanh (xu hướng, mùa vụ, thiếu dữ liệu)

Vẽ chuỗi thời gian, rolling statistics, kiểm tra khoảng trống thời gian, phát hiện outlier để quyết định cách làm sạch phù hợp.

In [ ]:
# EDA nhanh theo category lớn nhất
category_totals = monthly_sales.groupby("category", as_index=False)["y"].sum().sort_values("y", ascending=False)
top_categories = category_totals.head(3)["category"].tolist()

fig, ax = plt.subplots(figsize=(14, 5))
for category in top_categories:
    series = monthly_sales[monthly_sales["category"] == category].sort_values("ds")
    rolling = series["y"].rolling(window=3, min_periods=1).mean()
    ax.plot(series["ds"], series["y"], alpha=0.35, label=f"{category} - raw")
    ax.plot(series["ds"], rolling, linewidth=2)

ax.set_title("Doanh thu tháng của 3 category lớn nhất")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Revenue")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Kiểm tra missing months / outlier cơ bản
for category in top_categories:
    series = monthly_sales[monthly_sales["category"] == category].sort_values("ds")
    all_months = pd.date_range(series["ds"].min(), series["ds"].max(), freq="MS")
    missing_months = len(all_months.difference(series["ds"]))
    print(f"{category}: missing months = {missing_months}, min = {series['y'].min():.0f}, max = {series['y'].max():.0f}")

In [ ]:
prophet_frames = {}
for category in top_categories:
    series = monthly_sales[monthly_sales["category"] == category][["ds", "y"]].sort_values("ds").copy()
    full_months = pd.date_range(series["ds"].min(), series["ds"].max(), freq="MS")
    series = series.set_index("ds").reindex(full_months).fillna(0.0).reset_index()
    series.columns = ["ds", "y"]

    split_point = len(series) - 12
    train_df = series.iloc[:split_point].copy()
    valid_df = series.iloc[split_point:].copy()
    prophet_frames[category] = {"full": series, "train": train_df, "valid": valid_df}

    print(f"{category}: train={len(train_df)}, valid={len(valid_df)}")

## 5. Huấn luyện mô hình Prophet baseline

Khởi tạo Prophet với cấu hình mặc định, fit trên train, tạo future dataframe và sinh forecast baseline.

In [ ]:
baseline_results = []
baseline_forecasts = {}

for category in top_categories:
    frames = prophet_frames[category]
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    model.fit(frames["train"])

    future = model.make_future_dataframe(periods=12, freq="MS")
    forecast = model.predict(future)
    baseline_forecasts[category] = forecast

    validation_pred = forecast[forecast["ds"].isin(frames["valid"]["ds"])]
    merged = frames["valid"].merge(validation_pred[["ds", "yhat"]], on="ds", how="left")
    merged["abs_error"] = (merged["y"] - merged["yhat"]).abs()
    merged["sq_error"] = (merged["y"] - merged["yhat"]) ** 2
    merged["ape"] = np.where(merged["y"] != 0, merged["abs_error"] / merged["y"], np.nan)

    baseline_results.append(
        {
            "category": category,
            "mae": merged["abs_error"].mean(),
            "rmse": math.sqrt(merged["sq_error"].mean()),
            "mape_pct": merged["ape"].mean() * 100,
        }
    )

baseline_metrics = pd.DataFrame(baseline_results)
baseline_metrics

## 6. Đánh giá mô hình bằng backtesting và metrics

Dùng rolling validation đơn giản để tính MAE, RMSE, MAPE và so sánh giữa train-validation.

In [ ]:
# Tổng hợp metrics và lưu backtest summary
backtest_summary = baseline_metrics.copy()
backtest_summary["mape_pct"] = backtest_summary["mape_pct"].round(2)
backtest_summary["mae"] = backtest_summary["mae"].round(2)
backtest_summary["rmse"] = backtest_summary["rmse"].round(2)

print("Backtesting summary:")
backtest_summary

In [ ]:
param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1],
    "seasonality_prior_scale": [5.0, 10.0],
    "holidays_prior_scale": [5.0, 10.0],
    "seasonality_mode": ["additive", "multiplicative"],
}

best_params = {}
best_rows = []

for category in top_categories:
    frames = prophet_frames[category]
    best_mape = float("inf")
    best_combo = None

    for params in ParameterGrid(param_grid):
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            **params,
        )
        model.fit(frames["train"])
        forecast = model.predict(frames["valid"][ ["ds"] ])
        merged = frames["valid"].merge(forecast[["ds", "yhat"]], on="ds", how="left")
        ape = np.where(merged["y"] != 0, (merged["y"] - merged["yhat"]).abs() / merged["y"], np.nan)
        mape = np.nanmean(ape) * 100

        if mape < best_mape:
            best_mape = mape
            best_combo = params

    best_params[category] = best_combo
    best_rows.append({"category": category, "best_mape_pct": round(best_mape, 2), **best_combo})

best_tuning = pd.DataFrame(best_rows)
best_tuning

## 8. Thêm holiday và biến ngoại sinh (regressors)

Xây dựng bảng holiday tùy biến, thêm regressors (nếu có), refit mô hình và kiểm tra mức cải thiện hiệu năng.

In [ ]:
holiday_rows = []
for year in sorted(monthly_sales["ds"].dt.year.unique()):
    holiday_rows.extend(
        [
            {"holiday": "year_end_sale", "ds": pd.Timestamp(year=year, month=12, day=1)},
            {"holiday": "back_to_school", "ds": pd.Timestamp(year=year, month=8, day=1)},
            {"holiday": "summer_sale", "ds": pd.Timestamp(year=year, month=6, day=1)},
        ]
    )

holiday_df = pd.DataFrame(holiday_rows)

regressor_results = []
for category in top_categories:
    frames = prophet_frames[category]
    train_df = frames["train"].copy()
    valid_df = frames["valid"].copy()
    train_df["is_q4"] = train_df["ds"].dt.month.isin([10, 11, 12]).astype(int)
    valid_df["is_q4"] = valid_df["ds"].dt.month.isin([10, 11, 12]).astype(int)

    tuned = best_params.get(category, {})
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        holidays=holiday_df,
        **tuned,
    )
    model.add_regressor("is_q4")
    model.fit(train_df)

    future = model.make_future_dataframe(periods=12, freq="MS")
    future["is_q4"] = future["ds"].dt.month.isin([10, 11, 12]).astype(int)
    forecast = model.predict(future)
    validation_pred = forecast[forecast["ds"].isin(valid_df["ds"])]
    merged = valid_df.merge(validation_pred[["ds", "yhat"]], on="ds", how="left")
    ape = np.where(merged["y"] != 0, (merged["y"] - merged["yhat"]).abs() / merged["y"], np.nan)
    regressor_results.append({"category": category, "mape_pct": round(np.nanmean(ape) * 100, 2)})

regressor_metrics = pd.DataFrame(regressor_results)
regressor_metrics

## 9. Dự báo tương lai và trực quan hóa thành phần

Sinh dự báo cho horizon mục tiêu, vẽ forecast và components (trend/weekly/yearly), trích xuất `yhat`, `yhat_lower`, `yhat_upper`.

In [ ]:
final_forecast_rows = []

for category in top_categories:
    frames = prophet_frames[category]
    tuned = best_params.get(category, {})
    train_df = frames["full"].copy()
    train_df["is_q4"] = train_df["ds"].dt.month.isin([10, 11, 12]).astype(int)

    final_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        holidays=holiday_df,
        **tuned,
    )
    final_model.add_regressor("is_q4")
    final_model.fit(train_df)

    future = final_model.make_future_dataframe(periods=12, freq="MS")
    future["is_q4"] = future["ds"].dt.month.isin([10, 11, 12]).astype(int)
    forecast = final_model.predict(future)

    forecast["category"] = category
    final_forecast_rows.append(forecast[["category", "ds", "yhat", "yhat_lower", "yhat_upper"]].tail(12))

    fig = final_model.plot(forecast)
    fig.suptitle(f"Sales Forecast: {category}")
    plt.show()

    fig2 = final_model.plot_components(forecast)
    fig2.suptitle(f"Forecast Components: {category}")
    plt.show()

final_forecast = pd.concat(final_forecast_rows, ignore_index=True)
final_forecast.head()

## 10. Lưu mô hình và xuất artifact cho phase 3

Lưu model, lưu forecast và metrics ra file, ghi cấu hình tham số cuối cùng để tái lập và bàn giao phase 3.

In [ ]:
import joblib

artifact_dir = PROJECT_ROOT / "data" / "processed"
joblib.dump({"best_params": best_params, "holiday_df": holiday_df}, artifact_dir / "phase3_prophet_artifacts.joblib")
final_forecast.to_csv(artifact_dir / "demand_forecast_12m_notebook.csv", index=False)
best_tuning.to_csv(artifact_dir / "demand_forecast_tuning.csv", index=False)
regressor_metrics.to_csv(artifact_dir / "demand_forecast_regressor_metrics.csv", index=False)
backtest_summary.to_csv(artifact_dir / "demand_forecast_backtest_summary.csv", index=False)

print("Đã lưu toàn bộ artifact Phase 3.")
print(final_forecast.head())

## 7. Tinh chỉnh tham số Prophet

Thử grid search cho `changepoint_prior_scale`, `seasonality_prior_scale`, `holidays_prior_scale`, `seasonality_mode` và chọn cấu hình tốt nhất theo MAPE.

## 4. Chuẩn hóa dữ liệu cho Prophet (`ds`, `y`) và tách train/validation

Đổi tên cột theo chuẩn Prophet, bảo đảm tần suất dữ liệu nhất quán, chia tập train/validation theo mốc thời gian để tránh leak.

## 2. Nạp dữ liệu chuỗi thời gian và tiền xử lý

Đọc dữ liệu từ CSV/Parquet, ép kiểu thời gian, sắp xếp theo thời gian, xử lý trùng lặp, missing và nhiễu cơ bản trước khi modeling.

In [ ]:
# Import thư viện
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
POWERBI_DIR = PROJECT_ROOT / "powerbi"
POWERBI_DIR.mkdir(exist_ok=True, parents=True)

print("Đã cấu hình môi trường xong.")